# ML-07 — Baseline Action Score and Top-10 Review

**Lane (locked):** Refresh / Content Opportunity Scoring  
**Decision:** Which content pages should an editor review first?

This notebook does exactly three required things:

1. checks two real signals before using them,
2. builds one transparent baseline rule with a score + one reason code + action label,
3. reviews the top 10 with a skeptical “what would make this wrong?” note.

**Important:** `trend_direction` / `is_declining_label` are used only to audit whether a signal is associated with decline. They are **never inputs to the baseline score**.

In [ ]:
# SETUP — works from your repo or from an uploaded Colab notebook.

from pathlib import Path
import os
import subprocess
import json
import numpy as np
import pandas as pd

REPO_DIR = Path("/content/flyrank-ml-internship")
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

# If the notebook was uploaded directly to Colab, open the user's public repo.
if not DATA_REL.exists():
    if not REPO_DIR.exists():
        subprocess.run(
            [
                "git", "clone", "-q",
                "https://github.com/JaberAhmad555/flyrank-ml-internship.git",
                str(REPO_DIR)
            ],
            check=True
        )
    os.chdir(REPO_DIR)

assert DATA_REL.exists(), (
    "Could not find data/raw/content_refresh_anonymized.csv. "
    "Open this notebook from the FlyRank repo or restore the starter data file."
)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUT = OUTPUT_DIR / "baseline_action_score.csv"
JSON_OUT = OUTPUT_DIR / "baseline_action_score_metrics.json"

print("Repo root:", Path.cwd())
print("Data found:", DATA_REL)
print("Queue output:", CSV_OUT)
print("Metrics output:", JSON_OUT)

In [ ]:
# Load the small anonymized starter dataset.

df = pd.read_csv(DATA_REL)

# Outcome label ONLY for signal checking / evaluation.
# It is not used by the rule.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# Make sure the rule's numeric inputs are clean.
for col in ["days_since_last_update", "impressions_90d"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("Rows:", len(df))
print("Declining rate (audit only):", f"{df['is_declining_label'].mean():.2%}")
print("Rule inputs:", ["days_since_last_update", "impressions_90d"])
display(df[
    ["content_id", "client_id", "days_since_last_update",
     "impressions_90d", "is_declining_label"]
].head())

## 1. Check two signals first

### Signal A — staleness (`days_since_last_update`)
This is directly connected to FlyRank refresh logic: an older page may deserve review, especially when it still receives search visibility.

I will bucket pages by staleness, print **n**, and compare the observed decline rate.

### Signal B — visibility (`impressions_90d`)
This is the second signal my rule uses. High visibility increases the cost of ignoring a page, so I check whether the relationship is clean enough to justify using visibility as a **priority** signal.

The one-word verdicts are produced from the real bucket tables below: **CONFIRMED, OPPOSITE, MIXED, or FALSE**.

In [ ]:
# SIGNAL A — STALENESS BUCKET TABLE

staleness_order = ["0-89 days", "90-179 days", "180-364 days", "365+ days"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 89, 179, 364, np.inf],
    labels=staleness_order
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_days_since_update=("days_since_last_update", "median"),
          decline_rate=("is_declining_label", "mean"),
      )
      .reset_index()
)

display(staleness_table)

valid_stale = staleness_table[staleness_table["n"] > 0].copy()
rates = valid_stale["decline_rate"].to_numpy()

if len(rates) >= 2:
    delta = rates[-1] - rates[0]
    mostly_increasing = np.mean(np.diff(rates) >= -0.01) >= 0.67
    if delta >= 0.03 and mostly_increasing:
        staleness_verdict = "CONFIRMED"
    elif delta <= -0.03:
        staleness_verdict = "OPPOSITE"
    elif abs(delta) < 0.01 and (rates.max() - rates.min()) < 0.02:
        staleness_verdict = "FALSE"
    else:
        staleness_verdict = "MIXED"
else:
    staleness_verdict = "MIXED"

print("STALENESS VERDICT:", staleness_verdict)

In [ ]:
# SIGNAL B — VISIBILITY BUCKET TABLE
# Quantile buckets keep n reasonably balanced.

visibility_source = np.log1p(df["impressions_90d"].clip(lower=0))

try:
    df["visibility_bucket"] = pd.qcut(
        visibility_source,
        q=4,
        labels=["Low", "Medium", "High", "Very high"],
        duplicates="drop"
    )
except ValueError:
    # Fallback if repeated values make four quantiles impossible.
    df["visibility_bucket"] = pd.cut(
        df["impressions_90d"],
        bins=[-1, 0, 99, 499, np.inf],
        labels=["Low", "Medium", "High", "Very high"]
    )

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          decline_rate=("is_declining_label", "mean"),
      )
      .reset_index()
)

display(visibility_table)

valid_vis = visibility_table[visibility_table["n"] > 0].copy()
vrates = valid_vis["decline_rate"].to_numpy()

if len(vrates) >= 2:
    vdelta = vrates[-1] - vrates[0]
    monotonic = np.mean(np.diff(vrates) >= -0.01) >= 0.67
    if vdelta >= 0.03 and monotonic:
        visibility_verdict = "CONFIRMED"
    elif vdelta <= -0.03:
        visibility_verdict = "OPPOSITE"
    elif abs(vdelta) < 0.01 and (vrates.max() - vrates.min()) < 0.02:
        visibility_verdict = "FALSE"
    else:
        visibility_verdict = "MIXED"
else:
    visibility_verdict = "MIXED"

print("VISIBILITY VERDICT:", visibility_verdict)

print("\nInterpretation:")
print(
    "- Staleness is tested as a possible refresh-risk signal.\n"
    "- Visibility is mainly a PRIORITY signal: even if it does not predict decline cleanly, "
    "a visible page can deserve earlier human review because more search exposure is at stake."
)

## 2. Encode ONE baseline rule

### Rule idea
A page should rise in the review queue when it is:

- **still visible** in search, and
- **stale** enough that a content review may be useful.

The score is intentionally simple and readable:

\[
	ext{baseline score} =
0.55 	imes 	ext{visibility percentile}
+
0.45 	imes 	ext{staleness percentile}
\]

This is not a model and it does not use the decline label.

### One reason code per row
Each row gets **exactly one** reason code:

- `STALE_VISIBLE_PRIORITY`
- `STALE_PRIORITY`
- `VISIBLE_PRIORITY`
- `GENERAL_REVIEW`

### Action label
Pages with score ≥ 0.70 become `review_for_refresh`; the rest become `monitor`.

The exact threshold is a transparent baseline choice, not a claim that 0.70 is universally optimal. The Week-5 model has to beat this baseline.

In [ ]:
# Build the transparent baseline action score.
# IMPORTANT: only current, observable signals are used.

safe_rule_inputs = ["impressions_90d", "days_since_last_update"]

# Safety guard: label-derived / future-looking fields must not enter the rule.
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}
assert forbidden_inputs.isdisjoint(safe_rule_inputs)

df["visibility_score"] = (
    np.log1p(df["impressions_90d"].clip(lower=0))
      .rank(pct=True, method="average")
)

df["staleness_score"] = (
    df["days_since_last_update"]
      .rank(pct=True, method="average")
)

df["baseline_action_score"] = (
    0.55 * df["visibility_score"]
    + 0.45 * df["staleness_score"]
).clip(0, 1)

def one_reason_code(row):
    stale = row["days_since_last_update"] >= 180
    visible = row["impressions_90d"] >= 500

    if stale and visible:
        return "STALE_VISIBLE_PRIORITY"
    if stale:
        return "STALE_PRIORITY"
    if visible:
        return "VISIBLE_PRIORITY"
    return "GENERAL_REVIEW"

df["reason_code"] = df.apply(one_reason_code, axis=1)

df["action_label"] = np.where(
    df["baseline_action_score"] >= 0.70,
    "review_for_refresh",
    "monitor"
)

df["baseline_rank"] = (
    df["baseline_action_score"]
      .rank(method="first", ascending=False)
      .astype(int)
)

queue_cols = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "visibility_score",
    "staleness_score",
]

queue = (
    df[queue_cols]
      .sort_values(["baseline_rank", "content_id"])
      .reset_index(drop=True)
)

# Assignment-required output.
queue.to_csv(CSV_OUT, index=False)

print("Rule inputs:", safe_rule_inputs)
print("Forbidden inputs used in score: NONE")
print("Rows in queue:", len(queue))
print("Top score:", round(queue["baseline_action_score"].max(), 4))
print("Median score:", round(queue["baseline_action_score"].median(), 4))
print("Review-for-refresh rows:", int((queue["action_label"] == "review_for_refresh").sum()))
print("Wrote:", CSV_OUT)

display(queue.head(10))

In [ ]:
# Write a small metrics receipt.
# Unlike the generated CSV, this JSON is useful to commit with the notebook.

metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "rows": int(len(queue)),
    "rule_inputs": safe_rule_inputs,
    "score_formula": {
        "visibility_score": 0.55,
        "staleness_score": 0.45,
    },
    "action_threshold": 0.70,
    "top_score": float(queue["baseline_action_score"].max()),
    "median_score": float(queue["baseline_action_score"].median()),
    "review_for_refresh_rows": int(
        (queue["action_label"] == "review_for_refresh").sum()
    ),
    "signal_verdicts": {
        "staleness": staleness_verdict,
        "visibility": visibility_verdict,
    },
}

with open(JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
print("\nWrote:", JSON_OUT)

## 3. Top-10 skeptical review

For every top-ten page I want three things in one line:

1. **action** — what I would do,
2. **why it is here** — which observed signals pushed it up,
3. **what would make it wrong** — the reason a human reviewer might reject the recommendation.

The review does **not** use the decline label to justify the ranking.

In [ ]:
def why_here(row):
    parts = []
    if row["impressions_90d"] >= 500:
        parts.append(f"high visibility ({int(row['impressions_90d']):,} impressions)")
    else:
        parts.append(f"visibility score {row['visibility_score']:.2f}")

    if row["days_since_last_update"] >= 180:
        parts.append(f"stale ({int(row['days_since_last_update'])} days since update)")
    else:
        parts.append(f"staleness score {row['staleness_score']:.2f}")

    return " + ".join(parts)

def what_makes_it_wrong(row):
    stale = row["days_since_last_update"] >= 180
    visible = row["impressions_90d"] >= 500

    if stale and visible:
        return (
            "wrong if the page is intentionally evergreen, the update timestamp is stale, "
            "or seasonality explains the current traffic pattern"
        )
    if stale and not visible:
        return (
            "wrong if low search exposure means the editorial payoff is too small "
            "to justify refresh work"
        )
    if visible and not stale:
        return (
            "wrong if strong visibility is healthy demand and the page is already current"
        )
    return (
        "wrong if neither signal represents a real editorial opportunity after manual review"
    )

top10 = queue.head(10).copy()
top10["review_line"] = top10.apply(
    lambda r: (
        f"{r['action_label']} — {why_here(r)}; "
        f"{what_makes_it_wrong(r)}."
    ),
    axis=1
)

display(
    top10[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "reason_code",
            "action_label",
            "impressions_90d",
            "days_since_last_update",
            "review_line",
        ]
    ]
)

print("TOP-10 REVIEW — one line each\n")
for _, row in top10.iterrows():
    print(
        f"#{int(row['baseline_rank'])} {row['content_id']}: "
        f"{row['review_line']}"
    )

## 4. Weak picks

Even a top-ten list can contain weak recommendations.

I treat ranks **8–10** as the weakest top-ten picks because they sit closest to the cutoff. I inspect which component is weaker and say why that matters instead of pretending every high-ranked row is equally convincing.

In [ ]:
weak = top10.tail(3).copy()

def weak_pick_note(row):
    if row["visibility_score"] < row["staleness_score"]:
        return (
            "Weaker because the recommendation is driven more by staleness than visibility; "
            "editorial payoff may be limited if search demand is modest."
        )
    if row["staleness_score"] < row["visibility_score"]:
        return (
            "Weaker because the recommendation is driven more by visibility than staleness; "
            "a popular but current page may not need a refresh."
        )
    return (
        "Both components are similar, so the pick depends heavily on the chosen hand-written weights."
    )

weak["weak_pick_note"] = weak.apply(weak_pick_note, axis=1)

display(
    weak[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "visibility_score",
            "staleness_score",
            "weak_pick_note",
        ]
    ]
)

for _, row in weak.iterrows():
    print(
        f"#{int(row['baseline_rank'])} {row['content_id']}: "
        f"{row['weak_pick_note']}"
    )

## 5. Self-check

This baseline is intentionally simple. Its job is to create a transparent benchmark that the Week-5 model must beat.

Required checks:

- two visible bucket tables with **n**
- at least one flag-linked signal (staleness)
- one-word verdict for both signals
- one score
- exactly one reason code per row
- one action label
- ranked queue written to `work/outputs/baseline_action_score.csv`
- ten skeptical review lines
- weak-pick review
- no `trend_direction`, `trend_pct`, or `is_declining_label` used as score inputs

In [ ]:
# FINAL AUTOMATIC CHECKS

checks = {
    "two_signal_tables_exist": (
        isinstance(staleness_table, pd.DataFrame)
        and isinstance(visibility_table, pd.DataFrame)
    ),
    "bucket_tables_have_n": (
        "n" in staleness_table.columns
        and "n" in visibility_table.columns
    ),
    "signal_verdicts_valid": (
        staleness_verdict in {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}
        and visibility_verdict in {"CONFIRMED", "OPPOSITE", "MIXED", "FALSE"}
    ),
    "score_exists": "baseline_action_score" in queue.columns,
    "one_reason_code_per_row": (
        queue["reason_code"].notna().all()
        and ~queue["reason_code"].astype(str).str.contains(r"\|", regex=True).any()
    ),
    "action_label_exists": "action_label" in queue.columns,
    "queue_has_all_rows": len(queue) == len(df),
    "csv_written": CSV_OUT.exists(),
    "metrics_json_written": JSON_OUT.exists(),
    "top10_has_10_rows": len(top10) == 10,
    "top10_review_complete": top10["review_line"].notna().all(),
    "no_label_input": forbidden_inputs.isdisjoint(safe_rule_inputs),
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

assert all(checks.values()), "One or more self-checks failed."

print("\nALL REQUIRED NOTEBOOK CHECKS PASSED.")
print("Final queue:", CSV_OUT)
print("Metrics receipt:", JSON_OUT)
print("\nSave/commit this notebook as:")
print("work/notebooks/w04_baseline_score.ipynb")